<a href="https://colab.research.google.com/github/NicolasRodrigues07/Sprint1_IA_CHATBOT/blob/main/Chatbot_GoodWe_Sprint2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GoodWe EV Chatbot — LLaMA 3.2 — ChargeGrid Intelligence

Projeto Sprint 2 — Desenvolvimento e Entrega do Chatbot GoodWe

**Técnicas utilizadas:** RAG (Retrieval-Augmented Generation), few-shot prompting, memória de histórico

**Modelo:** `meta-llama/Llama-3.2-1B-Instruct` via HuggingFace  
**Framework:** LangChain + LangGraph  
**API Key:** gerenciada via Google Colab Secrets (`HF_TOKEN`)

In [46]:
%pip install --quiet groq langchain langchain-community langchain-core pypdf langchain-text-splitters langgraph sentence-transformers langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.0 MB/s eta 0:00:00


In [47]:
import os
from google.colab import userdata, files
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_huggingface import HuggingFaceEmbeddings
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print('Login Groq OK')

Login Groq OK


In [48]:
def chamar_modelo(messages):
    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=messages,
        max_tokens=512,
        temperature=0.3,
    )
    return response.choices[0].message.content

print('Modelo carregado!')

Modelo carregado!


## Upload dos PDFs

Faça upload dos PDFs do projeto GoodWe (ex: `chargegrid.pdf`, manuais técnicos, etc.).

In [49]:
uploaded = files.upload()
pdf_files = [f for f in uploaded.keys() if f.endswith('.pdf')]
print(f'PDFs carregados: {pdf_files}')

Saving chargegrid_intelligence_proposta.pdf to chargegrid_intelligence_proposta (1).pdf
Saving datasheet.pdf to datasheet (1).pdf
Saving manual.pdf to manual (1).pdf
PDFs carregados: ['chargegrid_intelligence_proposta (1).pdf', 'datasheet (1).pdf', 'manual (1).pdf']


In [50]:
# Embeddings + vector store
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store = InMemoryVectorStore(embeddings)
print('Vector store pronto!')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store pronto!


In [51]:
# Carregando e indexando os PDFs
all_docs = []
for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    all_docs.extend(docs)
    print(f'{pdf}: {len(docs)} paginas')

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(all_docs)
print(f'Split em {len(all_splits)} chunks.')

document_ids = vector_store.add_documents(documents=all_splits)
print(f'Indexados {len(document_ids)} chunks. Pronto!')

chargegrid_intelligence_proposta (1).pdf: 2 paginas
datasheet (1).pdf: 2 paginas
manual (1).pdf: 70 paginas
Split em 124 chunks.
Indexados 124 chunks. Pronto!


## RAG + System Prompt + Few-Shot

O system prompt define o escopo do chatbot (ChargeGrid Intelligence / EV ChargeOps) e inclui exemplos few-shot para guiar o estilo das respostas.

In [52]:
FEW_SHOT_EXAMPLES = """
Exemplos de como responder:

Pergunta: O que é a ChargeGrid?
Resposta: A ChargeGrid é uma rede de carregadores de veículos elétricos interconectados que gerenciam a distribuição de energia de forma inteligente, evitando sobrecargas na rede elétrica.

Pergunta: Qual o protocolo de comunicação usado?
Resposta: O GoodWe HCA G2 utiliza o protocolo Modbus/LAN para comunicação com os sistemas de gestão da ChargeGrid.

Pergunta: O carregador funciona no Brasil?
Resposta: Sim. Os modelos da linha HCA G2 são compatíveis com redes de 220/380 Vac, atendendo perfeitamente o padrão elétrico brasileiro.
"""


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str
    history_list: list


def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state['question'])
    return {'context': retrieved_docs}


def generate(state: State):
    docs_content = '\n\n'.join(doc.page_content for doc in state['context'])

    messages = [
        {
            "role": "system",
            "content": (
                "Você é o assistente oficial da GoodWe para o EV Challenge 2026, especializado em "
                "ChargeGrid Intelligence e EV ChargeOps. "
                "Responda APENAS com base no contexto dos documentos fornecidos. "
                "Se a resposta não estiver no contexto, diga: 'Não encontrei essa informação nos documentos do projeto.' "
                "Seja direto, objetivo e responda sempre em português.\n\n"
                f"{FEW_SHOT_EXAMPLES}\n"
                f"Contexto dos documentos:\n{docs_content}"
            )
        }
    ]

    for pergunta, resposta in state.get('history_list', []):
        messages.append({"role": "user", "content": pergunta})
        messages.append({"role": "assistant", "content": resposta})

    messages.append({"role": "user", "content": state['question']})

    answer = chamar_modelo(messages)
    return {'answer': answer}


graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, 'retrieve')
graph = graph_builder.compile()

print('RAG pronto!')

RAG pronto!


## Testes — Modelo de Avaliação Sprint 2

Executando os 5 casos de teste definidos na Sprint 1 e registrando os resultados.

In [54]:
casos_de_teste = [
    "O que é a ChargeGrid Intelligence e qual problema ela resolve?",
    "Quais são as especificações elétricas do GoodWe HCA G2 para o mercado brasileiro?",
    "Como funciona o gerenciamento dinâmico de carga (DLM) na ChargeGrid?",
    "Quais protocolos de comunicação o carregador GoodWe utiliza para integração com a smart grid?",
    "O que acontece com o homem aranha no carro"
]

print('=' * 60)
print('RESULTADOS DOS TESTES — SPRINT 2')
print('=' * 60)

for i, pergunta in enumerate(casos_de_teste, 1):
    print(f'\n[TESTE {i}]')
    print(f'Pergunta: {pergunta}')

    result = graph.invoke({
        'question': pergunta,
        'history_list': [],
        'context': [],
        'answer': ''
    })

    print(f'Resposta: {result["answer"]}')
    print(f'Chunks recuperados: {len(result["context"])}')
    print('-' * 60)


RESULTADOS DOS TESTES — SPRINT 2

[TESTE 1]
Pergunta: O que é a ChargeGrid Intelligence e qual problema ela resolve?
Resposta: A ChargeGrid Intelligence é uma solução de ponta focada no gerenciamento automatizado da infraestrutura de recarga para o setor comercial. Ela resolve o problema da ausência crítica de mecanismos nativos e integrados em eletropostos comerciais para gerenciar a potência distribuída, registrar o ciclo completo de cada sessão de recarga e aplicar políticas robustas de tarifação e pagamento fluido.
Chunks recuperados: 4
------------------------------------------------------------

[TESTE 2]
Pergunta: Quais são as especificações elétricas do GoodWe HCA G2 para o mercado brasileiro?
Resposta: De acordo com os documentos fornecidos, os modelos da linha HCA G2 são compatíveis com redes de 220/380 Vac, atendendo perfeitamente o padrão elétrico brasileiro.
Chunks recuperados: 4
------------------------------------------------------------

[TESTE 3]
Pergunta: Como funcion

## Chatbot Interativo com Histórico

Digite `sair` para encerrar.  
Digite `/historico` para ver o histórico da conversa.  
Digite `/limpar` para apagar o histórico.

In [56]:
print('GoodWe Chatbot — EV Challenge 2026')
print('Digite sair para encerrar')
print('Digite /historico para ver o historico da conversa')
print('Digite /limpar para apagar o historico')

historico = []

while True:
    pergunta = input('\nVoce: ').strip()

    if pergunta.lower() in ['sair', 'exit']:
        print('Ate logo!')
        break

    if pergunta.lower() == '/historico':
        print('\n--- Historico da conversa ---')
        if not historico:
            print('(vazio)')
        else:
            for i, (p, r) in enumerate(historico, 1):
                print(f'[{i}] Voce: {p}')
                print(f'[{i}] Bot: {r}')
        continue

    if pergunta.lower() == '/limpar':
        historico = []
        print('Historico apagado!')
        continue

    if not pergunta:
        continue

    result = graph.invoke({
        'question': pergunta,
        'history_list': historico,
        'context': [],
        'answer': ''
    })

    resposta = result['answer']
    historico.append((pergunta, resposta))

    print(f'\nBot: {resposta}')
    print(f'(baseado em {len(result["context"])} trechos dos PDFs | {len(historico)} mensagem(ns) no historico)')

GoodWe Chatbot — EV Challenge 2026
Digite sair para encerrar
Digite /historico para ver o historico da conversa
Digite /limpar para apagar o historico

Voce: O que é a ChargeGrid Intelligence e qual problema ela resolve?

Bot: A ChargeGrid Intelligence é uma solução de ponta focada no gerenciamento automatizado da infraestrutura de recarga para o setor comercial. Ela resolve o problema de ausência crítica de mecanismos nativos e integrados em eletropostos comerciais para gerenciar a potência distribuída, registrar o ciclo completo de cada sessão de recarga e aplicar políticas robustas de tarifação e pagamento fluido.
(baseado em 4 trechos dos PDFs | 1 mensagem(ns) no historico)

Voce: Quais são as especificações elétricas do GoodWe HCA G2 para o mercado brasileiro?

Bot: De acordo com os documentos fornecidos, os modelos da linha HCA G2 são compatíveis com redes de 220/380 Vac, atendendo perfeitamente o padrão elétrico brasileiro.
(baseado em 4 trechos dos PDFs | 2 mensagem(ns) no hist